In [1]:
# generate_plagiarism_dataset_noisy.py
import pandas as pd
import random
import numpy as np
from tqdm import tqdm

# -----------------------------
# Parameters
# -----------------------------
NUM_SAMPLES = 50000  # Total sentence pairs
PLAGIARISM_PROB = 0.5  # 50% plagiarized
NOISE_PROB = 0.2      # 20% chance to add noise

# -----------------------------
# Word pool and templates
# -----------------------------
WORDS = [
    "machine", "learning", "data", "algorithm", "network", "system",
    "research", "analysis", "process", "information", "model", "method",
    "impact", "study", "automation", "development", "knowledge", "technology"
]

TEMPLATES = [
    "The {} is very important for {}.",
    "{} can significantly improve {} in {}.",
    "Many people believe that {} {} {}.",
    "The process of {} involves {} and {}.",
    "Research shows that {} has a strong impact on {}.",
    "In the field of {}, {} plays a critical role.",
    "{} is essential for {} in {}.",
    "{} affects {} more than {}.",
    "The relationship between {} and {} is {}.",
    "Understanding {} requires knowledge of {} and {}."
]

SYNONYMS = {
    "important": ["crucial", "significant", "vital"],
    "improve": ["enhance", "boost", "increase"],
    "process": ["procedure", "method", "technique"],
    "research": ["study", "analysis", "investigation"],
    "knowledge": ["understanding", "awareness", "expertise"],
    "affects": ["impacts", "influences", "changes"],
    "essential": ["critical", "key", "necessary"],
    "data": ["information", "figures", "stats"],
    "model": ["framework", "structure", "design"]
}

# -----------------------------
# Helper functions
# -----------------------------

def generate_original_sentence():
    """Generates a general original sentence"""
    template = random.choice(TEMPLATES)
    words = random.choices(WORDS, k=template.count("{}"))
    return template.format(*words)

def plagiarize_sentence(sentence):
    """Generates a plagiarized version using synonyms, shuffles, and noise"""
    words = sentence.split()
    new_words = []
    for w in words:
        lw = w.lower().strip(".,")
        if lw in SYNONYMS and random.random() < 0.5:
            new_words.append(random.choice(SYNONYMS[lw]))
        else:
            new_words.append(w)

    # Shuffle small parts
    if len(new_words) > 6 and random.random() < 0.3:
        idx = random.randint(1, len(new_words)-2)
        new_words[idx], new_words[idx-1] = new_words[idx-1], new_words[idx]

    sentence = " ".join(new_words)

    # Add random noise
    if random.random() < NOISE_PROB:
        sentence = add_noise(sentence)

    return sentence

def add_noise(sentence):
    """Add minor noise to make dataset realistic"""
    # Randomly remove a character
    if random.random() < 0.3:
        idx = random.randint(0, len(sentence)-1)
        sentence = sentence[:idx] + sentence[idx+1:]

    # Randomly swap adjacent characters
    if random.random() < 0.3 and len(sentence) > 2:
        idx = random.randint(0, len(sentence)-2)
        sentence = sentence[:idx] + sentence[idx+1] + sentence[idx] + sentence[idx+2:]

    # Random punctuation insertion
    if random.random() < 0.2:
        idx = random.randint(0, len(sentence)-1)
        sentence = sentence[:idx] + random.choice([",", ".", ";"]) + sentence[idx:]

    # Random extra spaces
    sentence = " ".join([w + " " if random.random() < 0.1 else w for w in sentence.split()])

    return sentence

# -----------------------------
# Dataset generation
# -----------------------------
texts1 = []
texts2 = []
labels = []

print("Generating noisy plagiarism dataset...")

for _ in tqdm(range(NUM_SAMPLES)):
    orig_sentence = generate_original_sentence()
    is_plagiarized = 1 if random.random() < PLAGIARISM_PROB else 0

    if is_plagiarized:
        new_sentence = plagiarize_sentence(orig_sentence)
    else:
        # Also add noise to original sentences to prevent 100% accuracy
        new_sentence = generate_original_sentence()
        if random.random() < NOISE_PROB:
            new_sentence = add_noise(new_sentence)

    texts1.append(orig_sentence)
    texts2.append(new_sentence)
    labels.append(is_plagiarized)

# -----------------------------
# Save to CSV
# -----------------------------
df = pd.DataFrame({
    "text1": texts1,
    "text2": texts2,
    "label": labels
})

df.to_csv("plagiarism_dataset_final_noisy.csv", index=False)
print("Noisy dataset saved as plagiarism_dataset_final_noisy.csv")
print(df.head())

Generating noisy plagiarism dataset...


100%|██████████| 50000/50000 [00:00<00:00, 137170.52it/s]


Noisy dataset saved as plagiarism_dataset_final_noisy.csv
                                               text1  \
0  The process of learning involves algorithm and...   
1  Research shows that algorithm has a strong imp...   
2  Many people believe that network network analy...   
3  In the field of algorithm, model plays a criti...   
4  Research shows that knowledge has a strong imp...   

                                               text2  label  
0  The process of information involves informatio...      0  
1  The relationship betwen algorithm and machine ...      0  
2  The process of algorithm involves learning and...      0  
3  the In field of algorithm, structure plays a c...      1  
4  Research shows that knowledge has a strong imp...      1  


In [ ]:
# TwinText Plagiarism Detection Model (GPU Enabled)
# SBERT + Cosine Similarity + Logistic Regression
# Added: Precision, Recall, F1 Score, Confusion Matrix

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from sentence_transformers import SentenceTransformer
import torch


# -----------------------------
# 1 Load Dataset
# -----------------------------

print("Loading dataset...")
df = pd.read_csv("plagiarism_dataset_final.csv")
print("Dataset shape:", df.shape)
print(df.head())

# -----------------------------
# 2 Train Test Split
# -----------------------------

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(train_df))
print("Testing samples:", len(test_df))

# -----------------------------
# 3 Load SBERT Model (GPU)
# -----------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = SentenceTransformer("all-mpnet-base-v2", device=device)

# -----------------------------
# 4 Generate Embeddings
# -----------------------------

print("Generating embeddings on GPU...")

train_text1_embeddings = model.encode(
    train_df["text1"].tolist(),
    show_progress_bar=True,
    device=device,
    convert_to_numpy=True
)

train_text2_embeddings = model.encode(
    train_df["text2"].tolist(),
    show_progress_bar=True,
    device=device,
    convert_to_numpy=True
)

test_text1_embeddings = model.encode(
    test_df["text1"].tolist(),
    show_progress_bar=True,
    device=device,
    convert_to_numpy=True
)

test_text2_embeddings = model.encode(
    test_df["text2"].tolist(),
    show_progress_bar=True,
    device=device,
    convert_to_numpy=True
)

# -----------------------------
# 5 Compute Cosine Similarity
# -----------------------------

print("Computing similarity scores...")

train_similarity = np.array([
    cosine_similarity([train_text1_embeddings[i]], [train_text2_embeddings[i]])[0][0]
    for i in range(len(train_text1_embeddings))
]).reshape(-1, 1)

test_similarity = np.array([
    cosine_similarity([test_text1_embeddings[i]], [test_text2_embeddings[i]])[0][0]
    for i in range(len(test_text1_embeddings))
]).reshape(-1, 1)

# -----------------------------
# 6 Train Logistic Regression
# -----------------------------

print("Training classifier...")
classifier = LogisticRegression()
classifier.fit(train_similarity, train_df["label"])

# -----------------------------
# 7 Predictions
# -----------------------------

predictions = classifier.predict(test_similarity)

# -----------------------------
# 8 Model Evaluation
# -----------------------------

accuracy = accuracy_score(test_df["label"], predictions)
precision = precision_score(test_df["label"], predictions)
recall = recall_score(test_df["label"], predictions)
f1 = f1_score(test_df["label"], predictions)

print("\nModel Evaluation Results\n")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

# -----------------------------
# 9 Confusion Matrix
# -----------------------------

matrix = confusion_matrix(test_df["label"], predictions)
print("\nConfusion Matrix:\n", matrix)

# -----------------------------
# 10 Detailed Classification Report
# -----------------------------

print("\nClassification Report:\n")
print(classification_report(test_df["label"], predictions))

# -----------------------------
# 11 Plagiarism Check Function
# -----------------------------

def check_plagiarism(text1, text2):
    emb1 = model.encode([text1], device=device, convert_to_numpy=True)
    emb2 = model.encode([text2], device=device, convert_to_numpy=True)

    similarity = cosine_similarity(emb1, emb2)[0][0]
    prediction = classifier.predict([[similarity]])[0]

    result = "Plagiarism Detected" if prediction == 1 else "No Plagiarism"
    return similarity, result

# -----------------------------
# 12 Example Test
# -----------------------------

print("\nExample Test...\n")
text_a = "Machine learning improves automation in industries"
text_b = "Machine learning enhances automation in industries"
score, result = check_plagiarism(text_a, text_b)

print("Text 1:", text_a)
print("Text 2:", text_b)
print("\nSimilarity Score:", score)
print("Prediction:", result)

# -----------------------------
# 13 Manual Text Input Test
# -----------------------------

def manual_test():
    print("\n--- Plagiarism Detection Manual Test ---")
    while True:
        text1 = input("\nEnter Text 1:\n")
        text2 = input("\nEnter Text 2:\n")
        score, result = check_plagiarism(text1, text2)
        print("\nSimilarity Score:", score)
        print("Prediction:", result)
        if input("\nCheck another pair? (y/n): ").lower() != "y":
            print("\nProgram Ended.")
            break

if __name__ == "__main__":
    manual_test()

Loading dataset...
Dataset shape: (50000, 3)
                                               text1  \
0  The process of learning involves algorithm and...   
1  Research shows that algorithm has a strong imp...   
2  Many people believe that network network analy...   
3  In the field of algorithm, model plays a criti...   
4  Research shows that knowledge has a strong imp...   

                                               text2  label  
0  The process of information involves informatio...      0  
1  The relationship betwen algorithm and machine ...      0  
2  The process of algorithm involves learning and...      0  
3  the In field of algorithm, structure plays a c...      1  
4  Research shows that knowledge has a strong imp...      1  
Training samples: 40000
Testing samples: 10000
Using device: cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

Batches:   0%|          | 0/1250 [00:00<?, ?it/s]

Batches:   0%|          | 0/1250 [00:00<?, ?it/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Computing similarity scores...
Training classifier...

Model Evaluation Results

Accuracy : 0.9479
Precision: 0.9481436642453591
Recall   : 0.9468063671166633
F1 Score : 0.947474543804819

Confusion Matrix:
 [[4780  257]
 [ 264 4699]]

Classification Report:

              precision    recall  f1-score   support

           0       0.95      0.95      0.95      5037
           1       0.95      0.95      0.95      4963

    accuracy                           0.95     10000
   macro avg       0.95      0.95      0.95     10000
weighted avg       0.95      0.95      0.95     10000


Example Test...

Text 1: Machine learning improves automation in industries
Text 2: Machine learning enhances automation in industries

Similarity Score: 0.96894085
Prediction: Plagiarism Detected

--- Plagiarism Detection Manual Test ---

Enter Text 2:
today weather is so windy

Similarity Score: 0.15086593
Prediction: No Plagiarism
